# Deliverable 3 - INF201

Group members: Rishanth Jeevahen and Djordje Pesut, GitHub repository: https://github.com/djolepe06-bit/Week46_deliverable3

## Task 1 : Update the Character class

In [1]:
# Object-oriented game simulator
# Turid Torheim, NMBU, 2026 (FIXED)

class Character:
    """Represents a character in the RPG"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10,
                 weapon: "Weapon | None" = None) -> None:
        """
        Initialise the Character

        Args:
            name (str): The name of the character.
            hp_max (int): Max hit points (default = 100)
            damage (int): Damage delt when character attacks (default = 10)
            weapon (Weapon | None): Optional equipped weapon (default = None)
        """
        self.name = name
        self._hp = hp_max
        self.hp_max = hp_max
        self.damage = damage
        self.weapon = weapon  # optional weapon (Task 3)

    def attack(self, target) -> None:
        """Attack and deal damage to target character.

        If a weapon is equipped, damage is rolled from the weapon,
        otherwise the default base damage is used.
        """
        if self.weapon is not None:
            amount = self.weapon.roll_damage()
        else:
            amount = self.damage
        target.take_damage(amount)

    def take_damage(self, amount:int):
        self._hp = max(self._hp - amount, 0)

    def heal(self, amount: int) -> None:
        """Increase own HP, never above hp_max"""
        self._hp = min(self._hp + amount, self.hp_max)

    def introduce(self) -> None:
        """Creates string that introduces character"""
        print(f"Hi, my name is {self.name}!")

    def __repr__(self) -> str:
        """Representation of character object"""
        return f"<Character name={self.name} hp={self.hp} damage={self.damage}>"

    def __str__(self) -> str:
        """String representation of character object"""
        return f"Character: {self.name} with {self.hp} HP and {self.damage} damage"

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe edits."""
        return self._hp

    @property
    def is_alive(self) -> bool:
        """Check if character is alive (hp > 0)"""
        return self._hp > 0


*TEST RUN*

In [2]:
# Test

if __name__ == "__main__":
    # Create two characters
    char1 = Character(name="Alice", hp_max=120, damage=15)
    char2 = Character(name="Bob", hp_max=100, damage=10)

    # Introduce characters
    char1.introduce()
    char2.introduce()

    # Display initial HP
    print(f"{char1.name} has {char1.hp} HP.")
    print(f"{char2.name} has {char2.hp} HP.")

    # Alice attacks Bob
    char1.attack(char2)
    print(f"After attack, {char2.name} has {char2.hp} HP.")

    # Bob attacks Alice
    char2.attack(char1)
    print(f"After attack, {char1.name} has {char1.hp} HP.")

Hi, my name is Alice!
Hi, my name is Bob!
Alice has 120 HP.
Bob has 100 HP.
After attack, Bob has 85 HP.
After attack, Alice has 110 HP.


# Task 2: Dice class

In [3]:
import random


class Dice:
    """A class representing an n-sided dice."""

    def __init__(self, sides: int = 6) -> None:
        if sides < 2:
            raise ValueError("A dice must have at least 2 sides.")
        self.sides = sides

    def roll(self) -> int:
        """Rolls the dice and returns a random integer between 1 and sides."""
        return random.randint(1, self.sides)

    def __repr__(self) -> str:
        """Returns string representation of the Dice object."""
        return f"Dice(sides={self.sides})"

    def __call__(self) -> int:
        """Allows rolling the dice by calling the instance directly, e.g. d20()."""
        return self.roll()

*TEST RUN*

In [4]:
d20 = Dice(sides=20)
print(d20)
print(d20())

Dice(sides=20)
13


# Task 3: Create items


In [5]:
class Weapon:
    """A weapon that uses a Dice for damage calculations."""

    def __init__(self, name: str, damage_dice: Dice) -> None:
        self.name = name
        self.damage_dice = damage_dice

    def roll_damage(self) -> int:
        """Rolls the weapon's dice and returns the damage."""
        return self.damage_dice.roll()

    def __repr__(self) -> str:
        return f"Weapon(name={self.name!r}, damage_dice={self.damage_dice!r})"

*TEST RUN*

In [6]:
sword = Weapon("Sword", Dice(8))
hero = Character("Hero", weapon=sword)
unarmed = Character("Peasant")
goblin = Character("Goblin")

hero.attack(goblin)
print(goblin.hp)  # 92 to 99 (d8 damage)

unarmed.attack(goblin)
print(goblin.hp)  # 10 less than before (default damage)

93
83


# TASK 4: Character specialization

In [7]:

class Warrior(Character):
    """Represents a warrior, a defensive specialist."""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, armor: int = 5) -> None:

        """
        Initialise the Warrior

        Args:
            name (str): The name of the warrior.
            hp_max (int): Max hit points (default = 100)
            damage (int): Damage dealt when warrior attacks (default = 10)
            armor (int): The amount of armor the warrior has (default = 5)
            """

        super().__init__(name, hp_max, damage)  # take_damage is set to 5 by default
        self.armor = armor

    def attack(self, target) -> None:
        super().attack(target)

    def take_damage(self, amount: int) -> None:
        """Warrior takes damage reduced by armor"""
        super().take_damage(max(0, amount - self.armor))


class Mage(Character):
    """Represents a mage, a spell-casting specialist."""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, mana: int = 30, mana_max: int = 30) -> None:
        """
        Initialise the Mage

        Args:
            name (str): The name of the mage.
            hp_max (int): Max hit points (default = 100)
            damage (int): Damage dealt when mage attacks (default = 10)
            mana (int): The amount of mana the mage has for casting spells (default = 30)
            mana_max (int): Max mana the mage can have (default = 30)
        """
        super().__init__(name, hp_max, damage)
        self.mana = mana
        self.mana_max = mana_max

    def attack(self, target)-> None:
        if self.mana >= 10:
            self.mana -= 10
            target.take_damage(self.damage * 2)

        else:
            # restore mana
            self.mana = min(self.mana + 5, self.mana_max)
            super().attack(target)
            # fall back to the base attack if not enough mana

*TEST RUN #2*

In [8]:
alice = Character("Alice", hp_max=120, damage=15)
conan = Warrior("Conan", hp_max=150, damage=20, armor=5)

conan.attack(alice)
print(alice.hp)    # 100
alice.attack(conan)
print(conan.hp)    # 140  (15 - 5 armor = 10 damage)

mage = Mage("Merlin", hp_max=80, damage=10)
mage.introduce()
dummy = Character("Dummy", hp_max=200)
dummy.introduce()

for _ in range(5):
    mage.attack(dummy)
    print(mage.mana, dummy.hp)

100
140
Hi, my name is Merlin!
Hi, my name is Dummy!
20 180
10 160
0 140
5 130
10 120


# Task 5: Running battles

In [9]:
# Battle class
class battle:
    """Battle between two characters."""

    def __init__(self, hero: Character, enemy: Character) -> None:
        self.hero = hero
        self.enemy = enemy

    def play_around(self)-> None:
        """Hero attacks enemy , and if enemy is still alive, enemy attacks hero """
        self.hero.attack(self.enemy)
        if self.enemy.is_alive:
            self.enemy.attack(self.hero)

    def simulate_battle(self) -> None:
        """Simulate a battle between hero and enemy until one is defeated."""
        round_no = 0
        while self.hero.is_alive and self.enemy.is_alive:
            round_no +=1
            self.play_around()
            print(f"------------------------- ROUND{round_no}!!! -------------------------") # make a gap inn output 
            print()
            print(f"Round {round_no}: {self.hero.name} {self.hero.hp} HP | "
                  f"{self.enemy.name} {self.enemy.hp} HP")

        winner = self.hero if self.hero.is_alive else self.enemy
        print(f"{winner.name} wins after {round_no} rounds!")
        return winner
    

*TEST RUN*

In [10]:
# Battle 1: Warrior vs Mage, no weapons
conan = Warrior("Conan", hp_max=150, damage=20, armor=5)
merlin = Mage("Merlin", hp_max=80, damage=10)

battle(conan, merlin).simulate_battle()

------------------------- ROUND1!!! -------------------------

Round 1: Conan 135 HP | Merlin 60 HP
------------------------- ROUND2!!! -------------------------

Round 2: Conan 120 HP | Merlin 40 HP
------------------------- ROUND3!!! -------------------------

Round 3: Conan 105 HP | Merlin 20 HP
------------------------- ROUND4!!! -------------------------

Round 4: Conan 105 HP | Merlin 0 HP
Conan wins after 4 rounds!


<Character name=Conan hp=105 damage=20>

In [11]:
# Battle 2: Warrior vs Mage, both with weapons
conan = Warrior("Conan", hp_max=150, damage=20, armor=5)
merlin = Mage("Merlin", hp_max=80, damage=10)
conan.weapon = Weapon("Sword", Dice(8))
merlin.weapon = Weapon("Staff", Dice(6))   # used when Merlin is out of mana

battle(conan, merlin).simulate_battle()

------------------------- ROUND1!!! -------------------------

Round 1: Conan 135 HP | Merlin 78 HP
------------------------- ROUND2!!! -------------------------

Round 2: Conan 120 HP | Merlin 71 HP
------------------------- ROUND3!!! -------------------------

Round 3: Conan 105 HP | Merlin 70 HP
------------------------- ROUND4!!! -------------------------

Round 4: Conan 105 HP | Merlin 69 HP
------------------------- ROUND5!!! -------------------------

Round 5: Conan 105 HP | Merlin 65 HP
------------------------- ROUND6!!! -------------------------

Round 6: Conan 90 HP | Merlin 57 HP
------------------------- ROUND7!!! -------------------------

Round 7: Conan 90 HP | Merlin 56 HP
------------------------- ROUND8!!! -------------------------

Round 8: Conan 90 HP | Merlin 55 HP
------------------------- ROUND9!!! -------------------------

Round 9: Conan 75 HP | Merlin 53 HP
------------------------- ROUND10!!! -------------------------

Round 10: Conan 75 HP | Merlin 45 HP
---

<Character name=Conan hp=30 damage=20>

In [12]:
# Battle 3: plain Character vs plain Character (one with a weapon)
alice = Character("Alice", hp_max=100, damage=10, weapon=Weapon("Dagger", Dice(4)))
bob = Character("Bob", hp_max=100, damage=10)

battle(alice, bob).simulate_battle()

------------------------- ROUND1!!! -------------------------

Round 1: Alice 90 HP | Bob 96 HP
------------------------- ROUND2!!! -------------------------

Round 2: Alice 80 HP | Bob 94 HP
------------------------- ROUND3!!! -------------------------

Round 3: Alice 70 HP | Bob 92 HP
------------------------- ROUND4!!! -------------------------

Round 4: Alice 60 HP | Bob 88 HP
------------------------- ROUND5!!! -------------------------

Round 5: Alice 50 HP | Bob 86 HP
------------------------- ROUND6!!! -------------------------

Round 6: Alice 40 HP | Bob 85 HP
------------------------- ROUND7!!! -------------------------

Round 7: Alice 30 HP | Bob 81 HP
------------------------- ROUND8!!! -------------------------

Round 8: Alice 20 HP | Bob 79 HP
------------------------- ROUND9!!! -------------------------

Round 9: Alice 10 HP | Bob 78 HP
------------------------- ROUND10!!! -------------------------

Round 10: Alice 0 HP | Bob 74 HP
Bob wins after 10 rounds!


<Character name=Bob hp=74 damage=10>